In [1]:
import json
import random

In [2]:
# Function to sample a specific number of tokens
def sample_tokens(data, token_limit):
    selected = []
    total_tokens = 0
    for item in data:
        if total_tokens + item['len'] <= token_limit:
            total_tokens += item['len']
            selected.append({k: v for k, v in item.items() if k != 'len'})
        else:
            # Add a truncated version of the last item
            remaining_tokens = token_limit - total_tokens
            truncated_item = {k: v for k, v in item.items() if k != 'len'}
            truncated_item['output'] = truncated_item['output'][:remaining_tokens]
            selected.append(truncated_item)
            break
    return selected

# Load the JSON data
# with open('/mbz/users/liyuan/LLaMA-Factory/data/openmathinstruct2_1M_len.json', 'r') as f:

with open('/mbz/users/liyuan/LLaMA-Factory/data/Infinity-Instruct_0625_len.json', 'r') as f:
    structured_data = json.load(f)
    
    
# with open('/mbz/users/liyuan/LLaMA-Factory/data/opencoder-sft_len.json', 'r') as f:
#     structured_data = json.load(f) 

item_num = 323_000
if len(structured_data) >= item_num:
    sampled_data = random.sample(structured_data, item_num)
else:
    raise ValueError(f"The dataset contains fewer than {item_num} items.")

# Calculate the total tokens
total_tokens = sum(item['len'] for item in sampled_data)

print(f"Total tokens for the sampled {item_num} items: {total_tokens}")

Total tokens for the sampled 323000 items: 174039959


In [13]:
experiment_name = "exp2"
model_name="Llama-3.1-8B"
# 5M
# base_token = 5_000_000
# token_limits = {
#     f"{base_token}_{model_name}_instr_optim": int(base_token *0.51288044),
#     f"{base_token}_{model_name}_math_optim": int(base_token * 0.25065322),
#     f"{base_token}_{model_name}_code_optim": int(base_token * 0.23646634), 
# }

# 20M
# base_token = 20_000_000
# token_limits = {
#     f"{base_token}_{model_name}_instr_optim": int(base_token *0.47629393),
#     f"{base_token}_{model_name}_math_optim": int(base_token * 0.28764629),
#     f"{base_token}_{model_name}_code_optim": int(base_token * 0.23605978), 
# }

# 200M
base_token = 200_000_000
token_limits = {
    f"{base_token}_{model_name}_instr_optim": int(base_token *0.48666725),
    f"{base_token}_{model_name}_math_optim": int(base_token *0.29281993),
    f"{base_token}_{model_name}_code_optim": int(base_token *0.22051281), 
}

## Start sample training data

In [14]:
domain = "code"

original_dataset_path = "/mbz/users/liyuan/LLaMA-Factory/data/opencoder-sft_len.json"
save_dir = f"/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/{experiment_name}"

with open(original_dataset_path, 'r') as f:
    structured_data = json.load(f)

random.shuffle(structured_data)

all_selected_items = []
for name, limit in token_limits.items():
    if domain in name:
        sampled_items = sample_tokens(structured_data, limit)
        all_selected_items.extend(sampled_items)

        # Save each sampled subset
        subset_path = f"{save_dir}/{base_token}_{domain}_{name}.json"
        with open(subset_path, 'w') as f:
            json.dump(sampled_items, f, indent=4)

        print(f"Sampled {len(sampled_items)} items for {name} with token limit {limit}.")

Sampled 95702 items for 200000000_Llama-3.1-8B_code_optim with token limit 44102562.


In [15]:
domain = "instr"

original_dataset_path = "/mbz/users/liyuan/LLaMA-Factory/data/Infinity-Instruct_0625_len.json"
save_dir = f"/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/{experiment_name}"

# 1. Load and shuffle the data
with open(original_dataset_path, 'r') as f:
    structured_data = json.load(f)

random.shuffle(structured_data)

all_selected_items = []
for name, limit in token_limits.items():
    if domain in name:
        sampled_items = sample_tokens(structured_data, limit)
        all_selected_items.extend(sampled_items)

        # Save each sampled subset
        subset_path = f"{save_dir}/{base_token}_{domain}_{name}.json"
        with open(subset_path, 'w') as f:
            json.dump(sampled_items, f, indent=4)

        print(f"Sampled {len(sampled_items)} items for {name} with token limit {limit}.")

Sampled 180460 items for 200000000_Llama-3.1-8B_instr_optim with token limit 97333450.


In [16]:
domain = "math"

original_dataset_path = "/mbz/users/liyuan/LLaMA-Factory/data/openmathinstruct2_1M_len.json"
save_dir = f"/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/{experiment_name}"

# 1. Load and shuffle the data
with open(original_dataset_path, 'r') as f:
    structured_data = json.load(f)

random.shuffle(structured_data)

all_selected_items = []
for name, limit in token_limits.items():
    if domain in name:
        sampled_items = sample_tokens(structured_data, limit)
        all_selected_items.extend(sampled_items)

        # Save each sampled subset
        subset_path = f"{save_dir}/{base_token}_{domain}_{name}.json"
        with open(subset_path, 'w') as f:
            json.dump(sampled_items, f, indent=4)

        print(f"Sampled {len(sampled_items)} items for {name} with token limit {limit}.")

Sampled 132255 items for 200000000_Llama-3.1-8B_math_optim with token limit 58563986.


## Store data name into data_info

In [17]:
dataset_info_path = "/mbz/users/liyuan/LLaMA-Factory/data/dataset_info.json"
with open(dataset_info_path, "r") as f:
    try:
        dataset_info = json.load(f)
    except json.JSONDecodeError:
        dataset_info = {}

for domain in ["instr", "math", "code"]:
    # dataset_name = f"{base_token}_{domain}_val"
    # output_path = f"data_mixing/{experiment_name}/{dataset_name}.json"
    # dataset_info[dataset_name] = {
    #     "file_name": output_path
    # }
    for name, size in token_limits.items():
        if domain in name:
            dataset_name = f"{base_token}_{domain}_{name}"
            
            output_path = f"data_mixing/{experiment_name}/{dataset_name}.json"
            print(dataset_name)
            dataset_info[dataset_name] = {
                "file_name": output_path
            }
        
    with open(dataset_info_path, "w") as f:
        json.dump(dataset_info, f, indent=2)

200000000_instr_200000000_Llama-3.1-8B_instr_optim
200000000_math_200000000_Llama-3.1-8B_math_optim
200000000_code_200000000_Llama-3.1-8B_code_optim


In [18]:
dataset_info

{'GSM8K_test': {'file_name': 'GSM8K_test.json'},
 'Instruct-SkillMix': {'file_name': 'Instruct-SkillMix.json'},
 'FOLIO_like_data': {'file_name': 'FOLIO_like_data.json'},
 'FOLIO_validation': {'file_name': 'FOLIO_validation.json'},
 'Tulu-v2': {'file_name': 'Tulu-v2.json'},
 'deductive_200': {'file_name': 'deductive_200.json'},
 'alpaca_platypus_cot': {'file_name': 'alpaca_platypus_cot.json'},
 'bbh_4o': {'file_name': 'bbh(GPT4o).json'},
 'lima': {'file_name': 'lima.json'},
 'GPT4_alpaca': {'file_name': 'GPT4_alpaca.json'},
 'tracking_shuffled_objects_three_objects': {'file_name': 'tracking_shuffled_objects_three_objects.json'},
 'tracking_shuffled_objects_five_objects': {'file_name': 'tracking_shuffled_objects_five_objects.json'},
 'tracking_shuffled_objects_seven_objects': {'file_name': 'tracking_shuffled_objects_seven_objects.json'},
 'logical_deduction_three_objects': {'file_name': 'logical_deduction_three_objects.json'},
 'logical_deduction_five_objects': {'file_name': 'logical_de